# Add Methylation to Merged Datasets

Adds methylation MB-selected probes to each of the 8 existing Clinical+RNA+CNV+MUT+PROT merged files.

**Restart kernel before running.**

**Input:** `MERGE/01_ultra_conservative.csv` ... `08_composite.csv`  
**Output:** same files updated with `METH_` columns + `merge_summary_6modalities.csv`

## Setup — Paths, Dataset Pairs & Best Methylation Config

In [3]:
"""
ADD METHYLATION: Add methylation features to existing Clinical+RNA+CNV+MUT+PROT merged datasets.

Script location: 01_Causal_feature_extraction/
Input/Output:    01_Causal_feature_extraction/MERGE/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd.parent if cwd.name == "MERGE" else cwd

METH_DIR   = SCRIPT_DIR / "Methylation"
FILT_DIR   = METH_DIR / "statistical_filtered"
MB_DIR     = METH_DIR / "mb_results"
MERGE_DIR  = SCRIPT_DIR / "MERGE"

print(f"Script dir:   {SCRIPT_DIR}")
print(f"Methylation dir: {METH_DIR.exists()} -> {METH_DIR}")
print(f"Methylation MB:  {MB_DIR.exists()} -> {MB_DIR}")
print(f"MERGE dir:    {MERGE_DIR.exists()}")

DATASET_PAIRS = {
    "01_ultra_conservative": "meth_1_ultra_conservative_1028probes",
    "02_conservative":       "meth_2_conservative_1512probes",
    "03_standard":           "meth_3_standard_2479probes",
    "04_fdr_significant":    "meth_4_fdr_significant_1000probes",
    "05_balanced":           "meth_5_balanced_1512probes",
    "06_correlation":        "meth_6_correlation_1512probes",
    "07_top_correlated":     "meth_7_top_correlated_200probes",
    "08_composite":          "meth_8_composite_1000probes",
}

# ---------------------------------------------------------------------------
# STEP 1: LOAD PROTEINS SUMMARY AND SELECT BEST CONFIG PER DATASET
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 1: SELECT BEST METHYLATION CONFIG PER DATASET (highest C-index)")
print("="*70)
print(f"Reading: {MB_DIR / 'summary_all_results.csv'}")

summary = pd.read_csv(MB_DIR / "summary_all_results.csv")
print(f"Summary shape: {summary.shape}")
print(f"Datasets in summary: {summary['dataset'].unique().tolist()}")

best_meth = (
    summary
    .sort_values("c_index", ascending=False)
    .groupby("dataset", sort=False)
    .first()
    .reset_index()
)[["dataset", "algorithm", "alpha", "c_index"]]

print(best_meth.to_string(index=False))

Script dir:   C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
Methylation dir: True -> C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation
Methylation MB:  True -> C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\mb_results
MERGE dir:    True

STEP 1: SELECT BEST METHYLATION CONFIG PER DATASET (highest C-index)
Reading: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Methylation\mb_results\summary_all_results.csv
Summary shape: (56, 15)
Datasets in summary: ['meth_1_ultra_conservative_1028probes', 'meth_2_conservative_1512probes', 'meth_3_standard_2479probes', 'meth_4_fdr_significant_1000probes', 'meth_5_balanced_1512probes', 'meth_6_correlation_1512probes', 'meth_7_top_correlated_200probes', 'meth_8_composite_1000probes']
                             dataset algorithm  alpha  c_index
     meth_7_top_correlated_200probes      IAMB   0.05 0.644164
   meth_4_fdr_significant_1000probes      MMMB   0.05 0.570225
   

## Helper Functions

In [5]:
# ---------------------------------------------------------------------------
# STEP 2: HELPERS
# ---------------------------------------------------------------------------
def normalize_id(sample_id):
    """TCGA-D8-A146-01A  ->  TCGA-D8-A146"""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def load_genes(dataset, algorithm, alpha):
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        p = MB_DIR / dataset / f"{algorithm}_alpha{fmt}_genes.txt"
        if p.exists():
            return [l.strip() for l in p.read_text().splitlines() if l.strip()]
    raise FileNotFoundError(
        f"Gene file not found: {MB_DIR / dataset / f'{algorithm}_alpha{alpha:.2f}_genes.txt'}"
    )


def load_methylation(meth_dataset, algorithm, alpha):
    candidates = list(FILT_DIR.glob(f"{meth_dataset}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No methylation file found for '{meth_dataset}'")
    meth_file = candidates[0]

    genes      = load_genes(meth_dataset, algorithm, alpha)
    prot       = pd.read_csv(meth_file, index_col=0)
    available  = [g for g in genes if g in prot.columns]
    missing_g  = [g for g in genes if g not in prot.columns]
    if missing_g:
        print(f"  Warning: {len(missing_g)} methylation probes not in file")
    if not available:
        raise ValueError(f"No methylation available in {meth_file.name}")
    prot = prot[available].copy()
    print(f"  Methylation loaded: {prot.shape}  ({meth_file.name})")

    tumor_mask = prot.index.str.contains(r"-01[A-Z]?$", regex=True)
    prot = prot[tumor_mask].copy()
    print(f"  After Primary Tumor filter: {len(prot)} samples")

    prot.index = [normalize_id(i) for i in prot.index]

    if prot.index.duplicated().any():
        n_pts = prot.index[prot.index.duplicated(keep=False)].nunique()
        print(f"  Averaging {n_pts} patients with duplicate samples")
        prot = prot.groupby(prot.index).mean()

    prot.columns = [f"METH_{c}" for c in prot.columns]
    return prot

print("\nHelper functions ready")


Helper functions ready


## Add Methylation to All 8 Datasets

In [7]:
# ---------------------------------------------------------------------------
# STEP 3: ADD METHYLATION TO ALL 8 DATASETS
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 3: ADD METHYLATION TO ALL 8 MERGED DATASETS")
print("="*70)

results = []

for short_name, meth_dataset in DATASET_PAIRS.items():
    merged_file = MERGE_DIR / f"{short_name}.csv"
    if not merged_file.exists():
        print(f"\nSkipping {short_name} — file not found")
        continue

    row = best_meth[best_meth["dataset"] == meth_dataset]
    if row.empty:
        print(f"\nNo methylation summary entry for {meth_dataset} — skipping")
        continue
    row = row.iloc[0]

    print(f"\n{'─'*70}")
    print(f"[{short_name}]")
    print(f"  Methylation dataset: {meth_dataset}")
    print(f"  Config: {row['algorithm']}  alpha={row['alpha']}  C-index={row['c_index']:.4f}")

    try:
        merged = pd.read_csv(merged_file, index_col=0)

        # drop any METH_ columns from a previous run
        meth_cols_existing = [c for c in merged.columns if c.startswith("METH_")]
        if meth_cols_existing:
            merged = merged.drop(columns=meth_cols_existing)
            print(f"  Dropped {len(meth_cols_existing)} existing METH_ columns")

        n_clin = sum(1 for c in merged.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in merged.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in merged.columns if c.startswith("CNV_"))
        n_mut  = sum(1 for c in merged.columns if c.startswith("MUT_"))
        print(f"  Existing: {merged.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  MUT_={n_mut}")

        prot = load_methylation(meth_dataset, row["algorithm"], float(row["alpha"]))

        common = sorted(set(merged.index) & set(prot.index))
        print(f"  Common patients: {len(common)}  "
              f"(merged={len(merged)}, proteins={len(prot)})")
        if not common:
            raise ValueError("No common patients")

        final = pd.concat([merged.loc[common], prot.loc[common]], axis=1)

        assert final.isna().sum().sum() == 0, \
            f"Missing values: {final.isna().sum().sum()}"

        n_meth = sum(1 for c in final.columns if c.startswith("METH_"))
        n_clin = sum(1 for c in final.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in final.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in final.columns if c.startswith("CNV_"))
        n_mut  = sum(1 for c in final.columns if c.startswith("MUT_"))
        print(f"  Final: {final.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  "
              f"MUT_={n_mut}  METH_={n_meth}  | no missing")

        final.to_csv(merged_file)
        print(f"  Saved: {merged_file.name}")

        results.append({
            "file":             short_name + ".csv",
            "n_samples":        final.shape[0],
            "n_clin":           n_clin,
            "n_rna":            n_rna,
            "n_cnv":            n_cnv,
            "n_mut":            n_mut,
            "n_meth":           n_meth,
            "n_total":          final.shape[1],
            "meth_dataset":     meth_dataset,
            "meth_algorithm":   row["algorithm"],
            "meth_alpha":       row["alpha"],
            "meth_c_index":     row["c_index"],
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()


STEP 3: ADD METHYLATION TO ALL 8 MERGED DATASETS

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  Methylation dataset: meth_1_ultra_conservative_1028probes
  Config: GSMB  alpha=0.05  C-index=0.4354
  Existing: (745, 344)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50
  Methylation loaded: (772, 50)  (meth_1_ultra_conservative_1028probes.csv)
  After Primary Tumor filter: 771 samples
  Common patients: 533  (merged=745, proteins=771)
  Final: (533, 394)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50  METH_=50  | no missing
  Saved: 01_ultra_conservative.csv

──────────────────────────────────────────────────────────────────────
[02_conservative]
  Methylation dataset: meth_2_conservative_1512probes
  Config: IAMB  alpha=0.05  C-index=0.4353
  Existing: (745, 344)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50
  Methylation loaded: (772, 50)  (meth_2_conservative_1512probes.csv)
  After Primary Tumor filter: 771 samples
  Common patients: 533  (merged=745, protein

## Summary

In [9]:
# ---------------------------------------------------------------------------
# STEP 4: SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 4: SUMMARY")
print("="*70)

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    df.to_csv(MERGE_DIR / "merge_summary_6modalities.csv", index=False)
    print(f"\nCLIN_ same:  {df['n_clin'].nunique()==1}  ({df['n_clin'].iloc[0]})")
    print(f"RNA_ varies: {df['n_rna'].nunique()>1}  ({df['n_rna'].min()}–{df['n_rna'].max()})")
    print(f"CNV_ varies: {df['n_cnv'].nunique()>1}  ({df['n_cnv'].min()}–{df['n_cnv'].max()})")
    print(f"MUT_ varies: {df['n_mut'].nunique()>1}  ({df['n_mut'].min()}–{df['n_mut'].max()})")
    print(f"METH_ varies:{df['n_meth'].nunique()>1}  ({df['n_meth'].min()}–{df['n_meth'].max()})")
    print(f"Total:        {df['n_total'].min()}–{df['n_total'].max()} features")
    print(f"Samples:      {df['n_samples'].min()}–{df['n_samples'].max()}")
    print(f"\nSaved: {MERGE_DIR / 'merge_summary_6modalities.csv'}")
else:
    print("No datasets processed successfully.")


STEP 4: SUMMARY
                     file  n_samples  n_clin  n_rna  n_cnv  n_mut  n_meth  n_total                         meth_dataset meth_algorithm  meth_alpha  meth_c_index
01_ultra_conservative.csv        533     144     50     50     50      50      394 meth_1_ultra_conservative_1028probes           GSMB        0.05      0.435384
      02_conservative.csv        533     144     50     50     50      50      394       meth_2_conservative_1512probes           IAMB        0.05      0.435296
          03_standard.csv        533     144     50     50     50      50      394           meth_3_standard_2479probes           GSMB        0.05      0.447834
   04_fdr_significant.csv        533     144     50     50     50      50      394    meth_4_fdr_significant_1000probes           MMMB        0.05      0.570225
          05_balanced.csv        533     144     50     50     50      50      394           meth_5_balanced_1512probes           GSMB        0.05      0.543444
       06_correla